# Illustration of the scaling law of a recurrent network.

The aim of this notebook is to illustrate how the training hyperparameters scale when they are increased, showing how modifying a hyperparameter can affect all others.

## The Hyperparameters

The hyperparameters that we will consider are:

- Epochs: the number of epochs to train the model.
- Sequence length: axes 1 of the input data, the length of the input sequences. (characters length)
- Number of sequences: axes 0 of the input data, i.e. the number of sequences in the dataset.


### Static Hyperparameters

The following hyperparameters will remain constant throughout the notebook:
- Batch size
- Learning rate (Cosine scheduler applied)
- Token dimension
- Split ratio (train/test)

## Experiment

The analysis are performed by varying each hyperparameter independently and observing the effect on the training process and model performance, how accuracy and loss scale with the changes, and most of all, how much other previously untouched hyperparameters must change in order to ripristinate an optimal performance.

The experiment starts with a baseline configuration of hyperparameters, with optimal accuracy and loss, and will end showing the co-variance matrix of the hyperparameters, showing how they are correlated and how they affect each other.

## The Training Problem

Given a language grammar, the LSTM will be able to classify sequences of characters as valid or invalid according to the grammar rules.

BNF Definition:

$$
\begin{array}{rcl}
\langle\mathit{string}\rangle   & \mathrel{::=} & \langle\mathit{term}\rangle \\
                              & \mid          & \langle\mathit{string}\rangle \mathbin{\texttt{+}} \langle\mathit{term}\rangle \\[2pt]
\langle\mathit{term}\rangle   & \mathrel{::=} & AB, ED, OK \\[2pt]
\end{array}
$$

In [60]:
"""Static Hyperparameters Configuration"""

BATCH_SIZE = 16
SPLIT_RATIO = 0.9
MAX_LR, MIN_LR = 1e-2, 1e-4

In [61]:
"""Model Architecture"""

from thorcino.activations import Sigmoid
from thorcino.layers.linear import Linear
from thorcino.layers.lstm import LSTM
from thorcino.layers.sequential import Sequential
from thorcino.losses import BinaryCrossEntropyLoss
from thorcino.optimizer import SGD
from thorcino.training.schedulers import CosineSchedule
from thorcino.training.trainer import Trainer

def get_trainer(epochs: int):
    model = Sequential(
        LSTM(
            in_feature=3,
            hidden_units=3,
            out_type='n_to_1',
        ),
        Linear(
            in_feature=3,
            out_feature=1,
        ),
        Sigmoid()
    )
    loss = BinaryCrossEntropyLoss()
    optimizer = SGD(model.parameters, MAX_LR)
    scheduler = CosineSchedule(MAX_LR, MIN_LR, epochs)
    trainer = Trainer(
        model,
        loss,
        optimizer,
        scheduler,
    )

    return trainer

In [62]:
"""Data Generation"""

import numpy as np
from examples.recurrent.helpers import generate_invalid_seqs, generate_valid_seqs, parse_vect, tokenize
from thorcino.dataset.dataset import DataLoader, TensorDataset
from thorcino.tensor import Tensor

def get_dataset(row: int, col: int) -> tuple[np.array, np.array]:
    X_valid = np.array(tokenize(generate_valid_seqs(row, col)))
    X_invalid = np.array(tokenize(generate_invalid_seqs(row, col*2)))
    X = np.append(X_valid, X_invalid, axis=0)
    np.random.shuffle(X)

    Y = np.array([np.array([parse_vect(seq)], dtype=np.float32) for seq in X])

    return X, Y

def split_dataset(X: np.ndarray, Y: np.ndarray) -> tuple[np.array, np.array, np.array, np.array]:
    assert X.shape[0] == Y.shape[0]

    train_len = int(SPLIT_RATIO*X.shape[0])

    X_train, X_test = X[:train_len], X[train_len:]
    Y_train, Y_test = Y[:train_len], Y[train_len:]

    return X_train, Y_train, X_test, Y_test

def preprocess(X: np.ndarray, Y: np.ndarray) -> tuple[DataLoader, DataLoader]:
    X_train, Y_train, X_test, Y_test = split_dataset(X, Y)
    train_dataset, test_dataset = TensorDataset(Tensor(X_train), Tensor(Y_train)), TensorDataset(Tensor(X_test), Tensor(Y_test))
    train_dataloader, test_dataloader = DataLoader(train_dataset, BATCH_SIZE, True), DataLoader(test_dataset, BATCH_SIZE, True)

    return train_dataloader, test_dataloader

## The Training Process

Different trials will be performed for each hyperparameter, increasing its value by a 40% factor each time, every trails metrics will be plotted and compared to the baseline configuration, showing how the hyperparameter affects the training process and model performance.

In [63]:

def run_experiment(epochs: int, eval_step: int, n_sequence: int, sequence_length: int) -> dict:
    X, Y = get_dataset(n_sequence, sequence_length)
    train_dl, test_dl = preprocess(X, Y)

    trainer = get_trainer(epochs)

    for e in range(epochs):
        _ = trainer.train_epoch(train_dl)
        
        if e%eval_step == 0:
            _ = trainer.eval(test_dl)

    return trainer.history

In [64]:
import matplotlib.pyplot as plt

def plot_metrics(title:str, metrics: dict, epochs: int, eval_step: int):
    fig, (loss_ax, accuracy_ax) = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle(title)

    loss_ax.plot(range(epochs), metrics['train_loss'], label="train")
    loss_ax.plot(range(0, epochs, eval_step), metrics['eval_loss'], label="test")
    loss_ax.set_title("Losses")
    loss_ax.set_xlabel("epoch")
    loss_ax.set_ylabel("loss")
    loss_ax.legend()

    accuracy_ax.set_title("Accuracy on test sequence")
    accuracy_ax.plot(range(0, epochs, eval_step), metrics['accuracy'], label="accuracy")
    accuracy_ax.set_xlabel("epoch")
    accuracy_ax.set_ylabel("accuracy")
    accuracy_ax.legend()

    fig.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

In [ ]:
"""Testing Scaling Law"""

hyperparams = {
    'epochs': [50, 100, 150],
    'sequence_length': [5, 10, 20],
    'number_of_sequence': [100, 200, 400]
}

keys = hyperparams.keys()
for epoch in hyperparams['epochs']:
    for seq_len in hyperparams['sequence_length']:
        for n_seq in hyperparams['number_of_sequence']:
            eval_step = int(epoch/10)
            metrics = run_experiment(epoch, eval_step, n_seq, seq_len)
            plot_metrics(f'EPOCHS={epoch}\nNUMBER_OF_SEQUENCES{n_seq}\nSEQUENCE_LENGTH={seq_len}', metrics, epoch, eval_step)